# OSUM audio feature extraction (Werewolf-Among-Us, Game3)

Replicates the exact OSUM call used in `MultiMind` (Zhang et al., ACM Multimedia 2025) / its companion code repo `CjangCjengh/onuw` (`start_osum_api.py` + `onuw/mm_utils.py`):
one model call per utterance produces **both** a transcription **and** a categorical vocal-tone label (one of 8: happy, sad, neutral, angry, surprise, disgust, fear, other), via a single Chinese prompt appended-tag trick.

This is the same sample game (`ONE NIGHT ULTIMATE WEREWOLF Retro 3 / Game3`) already processed locally with Whisper + audEERING's wav2vec2-large-robust (dimensional arousal/valence/dominance) — kept as a separate, independent result for comparison, not a replacement. Only the **discussion-phase** utterances are processed here (19 of the game's 35 total) — the night phase and pre-discussion small talk are dropped, both because the app's spoken night-phase instructions ("wake up", "close your eyes", ...) bleed into the same audio track as the players' voices, and because that portion isn't the phenomenon of interest for persuasion-strategy analysis anyway. See `filter_discussion_phase.py` in the `werewolf-among-us` branch.

**Before running:** Runtime -> Change runtime type -> GPU. OSUM's own README states inference needs **~20GB VRAM** — the free-tier T4 (15GB) may OOM. If it does, this needs Colab Pro (A100/L4) rather than the free tier.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Clone OSUM and install dependencies

In [ ]:
!git clone https://github.com/ASLP-lab/OSUM.git
%cd OSUM/OSUM
# Drop torch/torchaudio/numpy/torch_npu/deepspeed entirely (keep Colab's own
# preinstalled, CUDA-matched torch; deepspeed is training-only and needs a
# matching CUDA toolkit to build from source - not required for inference).
!sed -i -E '/^(torch_npu|torch|torchaudio|numpy|deepspeed)(==|$| )/d' requirements.txt
# Strip exact version pins from everything else (e.g. transformers==4.44.0)
# so pip resolves to whatever version actually has a prebuilt wheel for
# Colab's current Python, instead of forcing an old pin that may only be
# available as a source distribution (this is what broke building
# `tokenizers`, a Rust extension transformers==4.44.0 pulls in - Colab has
# no Rust toolchain to build it from source).
!sed -i -E 's/==[^ ]+//' requirements.txt
!cat requirements.txt
!pip install -r requirements.txt
!pip install huggingface_hub librosa soundfile
# use_lora: true in OSUM's config means peft gets imported, and the latest
# peft checks for torchao>=0.16.0 (just a feature-detection gate, not a real
# functional need here) - Colab's preinstalled torchao (0.10.0) fails that
# check with "Found an incompatible version of torchao". Upgrade it.
!pip install -U torchao

In [ ]:
# wenet/squeezeformer/conv2d.py imports Union/Optional/Tensor/_pair from
# torch.nn.modules.conv - this only worked because older torch's internal
# conv.py happened to still have those names in its own namespace (a leaky
# implementation detail from ITS OWN imports, not something it ever
# officially re-exported). Torch 2.11 (Colab's current version) cleaned up
# that internal module and no longer exposes them there, so the import
# fails: "cannot import name 'Union' from 'torch.nn.modules.conv'".
# Patch it to import each name from where it's actually defined.
# Idempotent: safe to re-run if the disk (and this patch) already survived
# a runtime/GPU-type change - skips if already patched instead of asserting.
conv2d_path = "wenet/squeezeformer/conv2d.py"
with open(conv2d_path, "r", encoding="utf-8") as f:
    content = f.read()

old_import = "from torch.nn.modules.conv import _ConvNd, _size_2_t, Union, _pair, Tensor, Optional"
new_import = (
    "from typing import Optional, Union\n"
    "from torch import Tensor\n"
    "from torch.nn.common_types import _size_2_t\n"
    "from torch.nn.modules.conv import _ConvNd\n"
    "from torch.nn.modules.utils import _pair"
)
if new_import in content:
    print("Already patched, skipping:", conv2d_path)
else:
    assert old_import in content, "expected import line not found - OSUM's code may have changed since this was written"
    content = content.replace(old_import, new_import)
    with open(conv2d_path, "w", encoding="utf-8") as f:
        f.write(content)
    print("Patched", conv2d_path)

In [ ]:
# llmasr_model.py builds the Qwen2 backbone via
# AutoModelForCausalLM.from_config(config, device_map=..., output_hidden_states=..., ...).
# from_config() (unlike from_pretrained()) instantiates the model directly as
# cls(config, **kwargs) - it doesn't accept arbitrary generation/runtime flags
# as constructor kwargs. transformers==4.44.0 apparently tolerated both
# unknown kwargs silently; the current version rejects them one at a time
# ("unexpected keyword argument 'device_map'", then the same for
# 'output_hidden_states'). output_hidden_states is actually a config
# attribute, not a constructor arg - the correct fix is setting it on the
# config object before from_config(), not passing it as a kwarg. device_map
# is dropped entirely since our own code already does model.to(device).
# Idempotent: safe to re-run if the disk (and this patch) already survived
# a runtime/GPU-type change - skips if already patched instead of asserting.
llmasr_path = "wenet/llm_asr/llmasr_model.py"
with open(llmasr_path, "r", encoding="utf-8") as f:
    content = f.read()

old_block = (
    "self.llama_model = AutoConfig.from_pretrained(llm_path)\n"
    "                self.llama_model = AutoModelForCausalLM.from_config(self.llama_model,\n"
    "                                                                    torch_dtype=torch.bfloat16,\n"
    "                                                                    device_map=\"auto\",\n"
    "                                                                    trust_remote_code=True,\n"
    "                                                                    output_hidden_states=True,)"
)
new_block = (
    "self.llama_model = AutoConfig.from_pretrained(llm_path)\n"
    "                self.llama_model.output_hidden_states = True\n"
    "                self.llama_model = AutoModelForCausalLM.from_config(self.llama_model,\n"
    "                                                                    torch_dtype=torch.bfloat16,\n"
    "                                                                    trust_remote_code=True,)"
)
if new_block in content:
    print("Already patched, skipping:", llmasr_path)
else:
    assert old_block in content, "expected block not found - OSUM's code may have changed since this was written"
    content = content.replace(old_block, new_block)
    with open(llmasr_path, "w", encoding="utf-8") as f:
        f.write(content)
    print("Patched", llmasr_path)

In [ ]:
# llmasr_model.py's generate() does:
#   outputs = self.llama_model.generate(inputs_embeds=embeds, ...)
#   output_text = self.tokenizer.batch_decode(outputs, ...)
# assuming generate() returns a plain tensor of token ids. The current
# transformers version returns a ModelOutput (dict-like, via .sequences)
# by default for this generation path instead, so batch_decode's internal
# _decode() sees something dict-like without an "input_ids" key and raises
# KeyError: 'input_ids'. Unwrap .sequences first when present, otherwise
# use outputs as-is (covers both old plain-tensor and new dict-like
# returns). Idempotent like the earlier patches.
llmasr_path = "wenet/llm_asr/llmasr_model.py"
with open(llmasr_path, "r", encoding="utf-8") as f:
    content = f.read()

old_line = "        output_text = self.tokenizer.batch_decode(outputs, add_special_tokens=False, skip_special_tokens=True)"
new_line = (
    "        if hasattr(outputs, \"sequences\"):\n"
    "            outputs = outputs.sequences\n"
    "        output_text = self.tokenizer.batch_decode(outputs, add_special_tokens=False, skip_special_tokens=True)"
)
if new_line in content:
    print("Already patched, skipping:", llmasr_path, "(generate return-type fix)")
else:
    assert old_line in content, "expected line not found - OSUM's code may have changed since this was written"
    content = content.replace(old_line, new_line)
    with open(llmasr_path, "w", encoding="utf-8") as f:
        f.write(content)
    print("Patched", llmasr_path, "(generate return-type fix)")

## 2. Download the OSUM checkpoint

In [ ]:
from huggingface_hub import hf_hub_download
import os

ckpt_path = hf_hub_download(repo_id="ASLP-lab/OSUM", filename="infer.pt")
os.makedirs("OSUM", exist_ok=True)  # matches the relative path start_osum_api.py expects: 'OSUM/infer.pt'
link_path = "OSUM/infer.pt"
if os.path.lexists(link_path):
    os.remove(link_path)
os.symlink(ckpt_path, link_path)  # symlink instead of copy - instant regardless of checkpoint size
print("Checkpoint ready at", link_path, "->", ckpt_path)

## 3. Rebuild the same sample game's audio (matches the local audEERING run)

Pulls the dialogue annotation JSON from our `werewolf-among-us` branch, downloads the same game video from the HF dataset, extracts audio, and re-creates the identical per-utterance segments.

In [ ]:
!apt-get -qq install -y ffmpeg > /dev/null
!wget -q https://raw.githubusercontent.com/praeploykiat/Multimind/werewolf-among-us/raw/train.json -O train.json

import json
GAME_VIDEO_NAME = "ONE NIGHT ULTIMATE WEREWOLF  Retro 3"
GAME_ID = "Game3"
data = json.load(open("train.json", encoding="utf-8"))
game = next(g for g in data if g["video_name"] == GAME_VIDEO_NAME and g["Game_ID"] == GAME_ID)
print(game["video_name"], game["Game_ID"], len(game["Dialogue"]), "utterances")

In [ ]:
from huggingface_hub import hf_hub_download as hf_dl

# Video path within the bolinlai/Werewolf-Among-Us HF dataset (Youtube subset).
# Verified filename: "Youtube/videos/{video_name}_{Game_ID}.mp4" (double space in
# "WEREWOLF  Retro 3" is part of the actual filename, not a typo)
video_path = hf_dl(
    repo_id="bolinlai/Werewolf-Among-Us",
    repo_type="dataset",
    filename=f"Youtube/videos/{game['video_name']}_{game['Game_ID']}.mp4",
)
!ffmpeg -y -i "{video_path}" -vn -acodec pcm_s16le -ar 16000 -ac 1 game3_audio.wav -loglevel error
print("audio extracted")

In [ ]:
def to_sec(t):
    parts = [int(p) for p in t.split(":")]
    while len(parts) < 3:
        parts.insert(0, 0)
    h, m, s = parts
    return h * 3600 + m * 60 + s

# Restrict to the discussion phase only, dropping night-phase + pre-discussion
# small talk. Both risk app-narrator audio contamination (its scripted
# "wake up" / "close your eyes" instructions bleed into the same audio
# track) and aren't the phenomenon of interest for persuasion-strategy
# analysis anyway - confirmed via ASR inspection of the local audEERING run,
# see filter_discussion_phase.py in the werewolf-among-us repo.
# Rule generalizes across all 199 games without per-game timestamps: keep
# from the first utterance whose annotation isn't ['No Strategy'] onward,
# since the dataset's own strategy labels already mark where real
# discussion starts.
def first_discussion_index(dialogue):
    for i, u in enumerate(dialogue):
        ann = u.get("annotation", [])
        if ann and ann != ["No Strategy"]:
            return i
    return 0

full_dialogue = game["Dialogue"]
cut = first_discussion_index(full_dialogue)
print(f"Dropping {cut} pre-discussion utterances (night phase + small talk); "
      f"keeping {len(full_dialogue) - cut} discussion-phase utterances "
      f"starting at Rec_Id {full_dialogue[cut]['Rec_Id']} ({full_dialogue[cut]['timestamp']})")
dialogue = full_dialogue[cut:]

duration = to_sec(game["endTime"]) - to_sec(game["startTime"])
times = [to_sec(d["timestamp"]) for d in dialogue]
windows = []
for i, d in enumerate(dialogue):
    start = times[i]
    end = times[i + 1] if i + 1 < len(dialogue) else duration
    windows.append((start, max(end, start + 0.5)))
print(windows[:5])

## 4. Load OSUM and run the exact prompt used by `onuw`

This mirrors `start_osum_api.py`'s `load_model_and_processor` / `do_decode`, and `onuw/mm_utils.py`'s `_call_transcription_api` prompt + regex parsing, run in-process instead of over HTTP.

In [ ]:
import torch, torchaudio, librosa, numpy as np, re, soundfile as sf
from gxl_ai_utils.utils import utils_file
from wenet.utils.init_tokenizer import init_tokenizer
from gxl_ai_utils.config.gxl_config import GxlNode
from wenet.utils.init_model import init_model

EMOTIONS = ['sad', 'anger', 'neutral', 'happy', 'surprise', 'fear', 'disgust', 'other']
# Matches one of the 5 official <TRANSCRIBE><EMOTION> templates in
# OSUM/conf/prompt_config.yaml exactly (onuw's own mm_utils.py had a typo
# here - "angry" instead of "anger" - which doesn't match any trained
# template; fixed to the verbatim official wording for reliability).
PROMPT = '将音频转录为文字，并在文本最后附加<情感>标签，标签类型涵盖：sad，anger，neutral，happy，surprise，fear，disgust，还有other。'

args = GxlNode({'checkpoint': 'OSUM/infer.pt'})
# NOTE: onuw's start_osum_api.py used 'examples/osum/conf/...', which matched
# an older layout of the OSUM repo. The repo has since been reorganized -
# OSUM's own current infer_runtime.py confirms the config now lives directly
# at 'conf/...' relative to the OSUM/OSUM directory we're in.
configs = utils_file.load_dict_from_yaml('conf/config_llm_huawei_base-version.yaml')
model, configs = init_model(args, configs)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
tokenizer = init_tokenizer(configs)
print('OSUM loaded on', device)

In [ ]:
def compute_feat(waveform_1d, sample_rate=16000):
    waveform = torch.from_numpy(waveform_1d).float()
    window = torch.hann_window(400)
    stft = torch.stft(waveform, 400, 160, window=window, return_complex=True)
    magnitudes = stft[..., :-1].abs() ** 2
    filters = torch.from_numpy(librosa.filters.mel(sr=sample_rate, n_fft=400, n_mels=80))
    mel_spec = filters @ magnitudes
    log_spec = torch.clamp(mel_spec, min=1e-10).log10()
    log_spec = torch.maximum(log_spec, log_spec.max() - 8.0)
    log_spec = (log_spec + 4.0) / 4.0
    return log_spec.transpose(0, 1)

def run_osum(wav_np, sample_rate=16000):
    feat = compute_feat(wav_np, sample_rate).unsqueeze(0).to(device)
    feat_lens = torch.tensor([feat.shape[1]], dtype=torch.int64).to(device)
    with torch.no_grad():
        res_text = model.generate(wavs=feat, wavs_len=feat_lens, prompt=PROMPT)[0]
    speech, tone = res_text.strip(), 'other'
    m = re.search(r'^(.*)<(.*)>', res_text.strip())
    if m:
        speech, tone = m.group(1), m.group(2)
    if tone not in EMOTIONS:
        tone = 'other'
    return speech, tone

In [ ]:
full_audio, sr = sf.read('game3_audio.wav', dtype='float32')
assert sr == 16000

results = []
for i, (d, (start, end)) in enumerate(zip(dialogue, windows)):
    seg = full_audio[int(start*sr):int(end*sr)]
    item = {
        'Rec_Id': d['Rec_Id'], 'speaker': d['speaker'], 'timestamp': d['timestamp'],
        'window_sec': [round(start,2), round(end,2)],
        'utterance_ground_truth': d['utterance'], 'annotation': d['annotation'],
    }
    if seg.size < sr * 0.3:
        item['osum_transcript'], item['osum_tone'] = '', 'other'
    else:
        item['osum_transcript'], item['osum_tone'] = run_osum(seg, sr)
    results.append(item)
    print(f"[{i+1}/{len(dialogue)}] {d['speaker']}: tone={item['osum_tone']} text={item['osum_transcript'][:40]!r}")

with open('game3_osum_features.json', 'w', encoding='utf-8') as f:
    json.dump({'game': {'video_name': game['video_name'], 'Game_ID': game['Game_ID'], 'duration_sec': duration},
               'utterances': results}, f, indent=2, ensure_ascii=False)
print('Saved game3_osum_features.json')

## 5. Download the result
Download `game3_osum_features.json` from the Colab file browser (left sidebar) and send it back so it can be merged alongside `game3_audio_features.json` (the audEERING/Whisper result) for comparison.